# EE 342 Final Project Lab Report  

**Team members:** Joanna Zhou, Yehoshua Luna, Noah Sanchez  

**Dataset used:** https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones  

## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

## Helper Classes

In [5]:
class HARDataLoader:
    """
    A management class for loading the UCI Human Activity Recognition Dataset.
    """
    
    def __init__(self, root_dir='data/raw'):
        """
        Initializes the loader with the root data directory.
        Uses pathlib for robust, cross-platform path handling. 
        
        Args:
            root_dir (str): Relative path to the raw data folder.
        """
        self.root = Path(root_dir)
        
    def _load_files_in_folder(self, folder_path, prefix, dataset_type):
        """
        Internal helper to combine X, Y, and Z axis text files into a 3D tensor.
        
        Args:
            folder_path (Path): Path to the 'Inertial Signals' directory.
            prefix (str): Signal type (e.g., 'total_acc' or 'body_gyro').
            dataset_type (str): 'train' or 'test'.
            
        Returns:
            torch.Tensor: A 3D tensor with shape (samples, 128, 3). [cite: 6]
        """
        axes = ['x', 'y', 'z']
        data = []
        
        for axis in axes:
            # Construct filename based on axis and split (e.g., total_acc_x_train.txt)
            file_name = f'{prefix}_{axis}_{dataset_type}.txt'
            file_path = folder_path / file_name
            
            if not file_path.exists():
                raise FileNotFoundError(f"Missing expected data file: {file_path}")
            
            # Load space-separated data into NumPy
            data_array = pd.read_csv(file_path, sep=r'\s+', header=None).values
            data.append(data_array)
            
        # Stack along the last dimension to create shape: (Samples, TimeSteps, Channels)
        # Convert to float32 for compatibility with PyTorch CNN layers
        return torch.tensor(np.stack(data, axis=-1), dtype=torch.float32)

    def get_split(self, split='train'):
        """
        Retrieves the accelerometer, gyroscope, and label tensors for a split.
        
        Args:
            split (str): The dataset partition to load ('train' or 'test').
            
        Returns:
            accel (Tensor): Total acceleration (including gravity). [cite: 8]
            gyro (Tensor): Body angular velocity. [cite: 6]
            labels (Tensor): 0-indexed activity labels (Long). [cite: 1, 3]
        """
        split_dir = self.root / split
        signal_dir = split_dir / 'Inertial Signals'
        
        # Load accelerometer data (total acceleration = body + gravity)
        accel = self._load_files_in_folder(signal_dir, 'total_acc', split)
        
        # Load gyroscope data
        gyro = self._load_files_in_folder(signal_dir, 'body_gyro', split)
        
        # Load labels from y_train.txt or y_test.txt
        label_file = split_dir / f'y_{split}.txt'
        y_raw = pd.read_csv(label_file, sep=r'\s+', header=None).values
        
        # Flatten to 1D and shift labels 1-6 down to 0-5 for CrossEntropyLoss
        labels = torch.tensor(y_raw.squeeze(), dtype=torch.long) - 1
        
        return accel, gyro, labels

## Loading Data

In [6]:
loader = HARDataLoader('data/raw') # initializes data loader to correct path

In [7]:
# Load training data
train_accel, train_gyro, train_labels = loader.get_split('train')

print(f"Train accel shape: {train_accel.shape}")
print(f"Train gyro shape: {train_gyro.shape}")
print(f"Train labels shape: {train_labels.shape}")

Train accel shape: torch.Size([7352, 128, 3])
Train gyro shape: torch.Size([7352, 128, 3])
Train labels shape: torch.Size([7352])


In [8]:
# Load test data
test_accel, test_gyro, test_labels = loader.get_split('test')

print(f"Test accel shape: {test_accel.shape}")
print(f"Test gyro shape: {test_gyro.shape}")
print(f"Test labels shape: {test_labels.shape}")

Test accel shape: torch.Size([2947, 128, 3])
Test gyro shape: torch.Size([2947, 128, 3])
Test labels shape: torch.Size([2947])
